# Review Workflows

Three complete review setups, from fastest to most detailed:

1. [Quick validator](#1.-Quick-validator) — is the model's label right? One click (or key) per clip.
2. [Quick label](#2.-Quick-label) — pick the species: 10 one-click buttons for the common ones, a dropdown for the rest.
3. [Soundscape regions](#3.-Soundscape-regions) — mark a region (fixed window or box) and tag every species calling in it.

All three use the demo detections and audio from the base config
([`config/layout-and-buttons.yaml`](annotator_config/config/layout-and-buttons.yaml)); each
writes its own output file and is set up so a clip is reviewed once (`duplicate_entries=False`):
submitted clips show their saved answers, and the last one shows an "all done" summary.

**Keys:** `k` / `j` next / previous clip · `Shift+Tab` moves between sections · `Tab` into the
form · a button's hot key works while its row / column is focused (click it or Tab to it).

In [1]:
from jupyter_bioacoustic import BioacousticAnnotator

CONFIG = 'annotator_config/config/layout-and-buttons.yaml'
SPECIES = 'data/species_counts.N-dse.csv'


def jba(**settings):
    """Open the annotator on the base config plus ``settings``; print any validation problems."""
    ba = BioacousticAnnotator(config=CONFIG, duplicate_entries=False, **settings)
    for msg in ba.validate()['errors']:
        print('❌', msg)
    ba.open()
    return ba

---
## 1. Quick validator

Is the detection right? The form sits beside the spectrogram: a title naming this clip's species
and a column of three icon buttons. Each one writes `is_correct` (`yes` / `no` / `unsure`),
submits and moves to the next clip — there is no separate Submit button.

Keys (with the button column focused): `y` yes · `n` no · `u` unsure.

In [2]:
validator = jba(
    output_path='outputs/quick-validator.csv',
    form_config={
        'side_form': True,
        'side_form_split': 0.8,
        'submission_buttons': False,
        'title': 'Is [[common_name]] correct?',
        'form': [
            {'column': {'column': 'is_correct', 'gap': 8, 'items': [
                {'button': {'value': 'yes', 'icon': 'jba:check', 'label': 'yes', 'hot_key': 'y', 'submit': True}},
                {'button': {'value': 'no', 'icon': 'jba:x', 'label': 'no', 'hot_key': 'n', 'submit': True}},
                {'button': {'value': 'unsure', 'icon': 'jba:unsure', 'label': 'unsure', 'hot_key': 'u', 'submit': True}},
            ]}},
        ],
    },
)

<IPython.core.display.Javascript object>

In [3]:
validator.output(force=True)

,detection_id,is_correct
0,5780,yes
1,1191,no
2,5429,yes
3,6456,unsure
4,5203,yes
5,1682,unsure
6,958,yes


---
## 2. Quick label

What species is it? The ten most common species in this dataset are one-click buttons (hot keys
`1`–`9`, `0`, shown on the buttons): each writes `label`, submits and moves on. For anything else,
pick it in the dropdown (type to filter) and press **Submit** — the only thing that needs it.

The detector's guess is in the title, so a quick look at the spectrogram is usually enough.

In [9]:
TOP_10 = [
    'Pacific-slope Flycatcher', 'Spotted Towhee', 'California Towhee', 'Tree Swallow',
    'Yellow-breasted Chat', "Bewick's Wren", 'Spotted Crake', 'Hermit Thrush',
    "Bell's Vireo", 'Great Horned Owl',
]
KEYS = '1234567890'

labeler = jba(
    output_path='outputs/quick-label.csv',
    form_config={
        'side_form': True,
        'side_form_split': 0.70,
        'title': {'value': 'Quick Label', 'progress_tracker': True},
        'form': [
            {'grid': {'columns': 2, 'gap': 6, 'column': 'label', 'items': [
                {'button': {'value': name, 'icon': name, 'width': 165, 'height': 28,
                            'hot_key': key, 'show_hot_key': True, 'submit': True}}
                for name, key in zip(TOP_10, KEYS)
            ]}},
            {'line': True},
            {'select': {'label': 'other species', 'column': 'label', 'required': True, 'width': 170,
                        'items': {'path': SPECIES, 'value': 'common_name', 'filter_box': True,
                                  'not_available': {'label': 'Unknown species', 'value': 'unknown'}}}},
        ],
    },
)

<IPython.core.display.Javascript object>

In [10]:
labeler.output(force=True)

,detection_id,label
0,5780,Pacific-slope Flycatcher
1,1191,California Towhee
2,5429,Yellow-breasted Chat
3,6456,Bewick's Wren


---
## 3. Soundscape regions

A dawn-chorus survey: for each clip, mark **one region** and tag **every species calling in it**
(multilabel), then describe the region.

- **Region** — pick a tool from the dropdown: **fixed duration** (click to drop an adjustable
  3 s window; good for overlapping chorus) or **bounding box** (drag a time × frequency box
  around one call). The bounds go to `start_time` / `end_time` (and `min_freq` / `max_freq` for a
  box).
- **Species present** — a multiselect: each pick becomes a tag (type to filter; `1`–`5` add / remove
  the most common species while it's focused — they're pinned at the top of the list). Saved to
  `species_present` as `name;name;...`.
- **Region details** — call types (any number, one column), how many individuals (one-hot
  `count_1` / `count_2` / `count_3plus`), signal quality, and a "needs expert" flag that asks why.

In [17]:
survey = jba(
    output_path='outputs/soundscape-regions.csv',
    player_height=300,
    form_config={
        'title': {'value': 'SOUNDSCAPE REGION — detector: [[common_name]]', 'progress_tracker': True},
        'annotation': {
            'start_time': {'column': 'start_time'},
            'end_time': {'column': 'end_time'},
            'min_frequency': {'column': 'min_freq'},
            'max_frequency': {'column': 'max_freq'},
            'tools': [
                {'fixed_duration': {'initial_window': 3, 'min': 1, 'max': 10, 'step': 0.5}},
                'bounding_box',
            ],
        },
        'form': [
            {'multiselect': {
                'label': 'species present', 'column': 'species_present', 'required': True,
                'items': {'path': SPECIES, 'value': 'common_name', 'filter_box': True},
                'hot_keys': {1: 'Pacific-slope Flycatcher', 2: 'Spotted Towhee',
                             3: 'California Towhee', 4: 'Tree Swallow', 5: 'Yellow-breasted Chat'},
                'pin_hot_keys': True,
                'output': {'format': 'list', 'delimiter': ';'},
            }},
            {'grid': {'columns': 2, 'gap': 10, 'items': [
                {'text': 'call types:'},
                {'row': {'column': 'call_types', 'unique': False, 'delimiter': ';', 'gap': 4, 'items': [
                    {'button': {'value': 'song', 'icon': 'song', 'width': 52}},
                    {'button': {'value': 'call', 'icon': 'call', 'width': 52}},
                    {'button': {'value': 'alarm', 'icon': 'alarm', 'width': 52}},
                    {'button': {'value': 'drum', 'icon': 'drum', 'width': 52}},
                ]}},
                {'text': 'individuals:'},
                {'row': {'unique': True, 'gap': 4, 'items': [
                    {'button': {'column': 'count_1', 'value': 1, 'initial_value': 0, 'icon': '1'}},
                    {'button': {'column': 'count_2', 'value': 1, 'initial_value': 0, 'icon': '2'}},
                    {'button': {'column': 'count_3plus', 'value': 1, 'initial_value': 0, 'icon': '3+'}},
                ]}},
                {'text': 'signal quality:'},
                {'select': {'label': '', 'column': 'quality', 'required': True,
                            'items': ['clear', 'faint', 'distant', 'masked by noise']}},
            ]}},
            {'row': [
                {'checkbox': {'label': 'needs expert review', 'column': 'needs_expert',
                              'checked_value': 'yes', 'unchecked_value': 'no',
                              'checked_form': 'expert'}},
                {'textbox': {'label': 'notes', 'column': 'notes', 'width': 260}},
            ]},
        ],
        'dynamic_forms': {
            'expert': [
                {'select': {'label': 'why?', 'column': 'expert_reason', 'required': True,
                            'items': ['unfamiliar call', 'possible rare species',
                                      'overlapping species', 'possible mimic', 'other']}},
            ],
        },
        'submission_buttons': {'previous': True, 'next': {'label': 'Skip'}, 'submit': {'label': 'Save region'}},
    },
)

<IPython.core.display.Javascript object>

In [18]:
survey.output(force=True)